# w10-2. 학습한 모델 저장하고 다시 사용하기

**오늘 할 일**
1. 왜 모델을 저장해야 하는지 이해
2. **Logistic Regression**을 예시로 저장 → 불러오기 → 새 분자 예측 (worked example)
3. 같은 방식으로 **ANN, SVM, Decision Tree, Random Forest** 저장하기 (✏️ Grid search를 통해 가장 좋은 하이퍼파라미터를 확인한 후, 모델을 저장)

**데이터**: `skin_irritation_2Ddesc.csv` (지난 시간과 동일)

## 🚨 핵심 포인트

| 모델 | 저장해야 할 것 |
|---|---|
| Logistic Regression / ANN / SVM | feature 이름 + **scaler** + 모델 |
| Decision Tree / Random Forest | feature 이름 + 모델 (scaler 불필요) |

> ⚠️ normalization을 쓴 모델은 **학습 때 fit한 scaler**를 그대로 다시 써야 해.
> 새 분자에 `fit_transform`을 새로 부르면 **다른 평균/표준편차**로 변환되어 모델이 망가져.

---
# Part A. 이제까지 여러가지 실험을 진행한 이유?

- **새로운 분자 구조**를 예측할 때 사용할, 모델을 확정짓기 위함.
- 여러가지 실험을 통해 모델 성능을 확인하고, 가장 성능이 높은 모델을 최종 모델로 확정 및 배포. (다른 사람들도 사용할 수 있게 공유)

파이썬에서 모델을 저장하는 표준 도구는 **`joblib`**.

```python
import joblib
joblib.dump(객체, '파일이름.joblib')   # 저장
객체 = joblib.load('파일이름.joblib')   # 불러오기
```

---
# Part B. Logistic Regression — worked example

## B-1. 데이터 준비 + 모델 학습

지난 시간(w9-1) 코드 그대로.

## 🚨 가장 중요한 약속 — 학습과 예측은 같은 전처리

Logistic Regression은 **normalization 된 feature**로 학습한다.
→ 그래서 나중에 **새 분자에 예측할 때도 반드시 normalization 된 feature**를 넣어야 한다.
→ normalization을 다시 fit하면 평균/표준편차가 달라져서 망가지므로, **학습 때 fit한 scaler를 그대로 재사용**해야 한다.

```
[학습] X_sel  --(scaler.fit_transform)-->  X_scaled  --(clf.fit)-->  모델
[예측] new_X  --(scaler.transform)------>  X_scaled  --(clf.predict)-->  결과
                  ⬆ 같은 scaler! fit 다시 하면 안 됨
```

그래서 학습이 끝나면 `cols`, **`scaler`**, `clf` 세 개를 모두 손에 들고 있어야 함. 하나라도 빠지면 새 분자 예측 불가능.

In [1]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 데이터 불러오기
df = pd.read_csv('skin_irritation_2Ddesc.csv')
y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

# NaN 열 제거 + 저분산 열 제거
X = X.dropna(axis=1)
X = X.loc[:, X.std() >= 0.01]

# 상위 10개 descriptor 선택
selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
cols = X.columns[selector.get_support()]   # 골라진 feature 이름들
X_sel = X[cols]
print('선택된 descriptor:', list(cols))

# ⭐ Normalization — Logistic Regression은 scale에 민감하므로 필수
# fit_transform: 학습 데이터의 평균/표준편차를 '계산(fit)'하고 동시에 '변환(transform)'
# scaler 객체에는 학습 데이터의 평균/표준편차가 저장됨 → 나중에 새 분자에 똑같이 적용해야 함
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sel)

# ⭐ 학습은 normalization 된 X_scaled로! (X_sel 원본 ❌)
clf = LogisticRegression(max_iter=5000)
clf.fit(X_scaled, y)
print('학습 정확도:', round(clf.score(X_scaled, y), 3))

선택된 descriptor: ['MinAbsEStateIndex', 'BertzCT', 'Chi0', 'Chi1', 'PEOE_VSA7', 'SlogP_VSA6', 'HeavyAtomCount', 'MolMR', 'fr_C_O_noCOO', 'fr_ester']
학습 정확도: 0.872


## B-2. 모델 저장하기

Logistic Regression은 normalization이 필요한 모델이라 **세 가지**를 같이 저장해야 해.

1. **`cols`** — SelectKBest가 고른 feature 이름. 새 분자에서 **같은 컬럼**을 같은 순서로 뽑아야 함.
2. **`scaler`** — 학습 때 fit된 StandardScaler. 학습 데이터의 평균/표준편차 값이 들어 있음.
3. **`clf`** — 학습된 LogisticRegression.

세 개를 **딕셔너리**로 묶으면 파일 하나로 깔끔하게 저장 가능.

In [2]:
import joblib

# 세 가지를 딕셔너리로 묶기
saved = {
    'features': list(cols),   # feature 이름 (리스트로 변환)
    'scaler': scaler,         # 학습 때 fit한 StandardScaler
    'model': clf              # 학습된 LogisticRegression
}

# 한 파일로 저장
joblib.dump(saved, 'model_logreg.joblib')
print('저장 완료: model_logreg.joblib')

#디스크립터 정보, 정규화했으면 새로운 데이터도 그렇게 할 수 있게, 모델 저장

저장 완료: model_logreg.joblib


## B-3. 모델 불러와서 잘 저장됐는지 확인

저장이 잘 됐는지 확인하는 가장 쉬운 방법:
**불러와서 같은 X에 다시 예측 → 학습 때 확인한 성능과 동일하면 성공.**

> ⚠️ 불러온 scaler는 **`transform`** 만 호출! `fit_transform`을 다시 부르면 평균/표준편차가 새로 계산되어 학습 때와 달라짐.

In [3]:
import joblib

# 파일 불러오기
loaded = joblib.load('model_logreg.joblib')
features = loaded['features']
scaler   = loaded['scaler']
model    = loaded['model']

print('저장된 feature 개수:', len(features))
print('저장된 모델 종류  :', type(model).__name__)

# 같은 X에 다시 예측 → 학습 때와 같은 정확도가 나와야 함
X_check = X[features]                          # 저장된 feature 이름으로 다시 추출
X_check_scaled = scaler.transform(X_check)     # ⚠️ transform만! fit_transform 아님
y_pred = model.predict(X_check_scaled)

print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))
print('학습 때 정확도와 같으면 저장 성공 ✅')

저장된 feature 개수: 10
저장된 모델 종류  : LogisticRegression
불러온 모델 정확도: 0.872
학습 때 정확도와 같으면 저장 성공 ✅


## B-4. 모델을 이용해서 새로운 분자구조 예측

학습된 모델로 **새로운 분자의 피부 자극 여부를 예측**해 보자.

## 🚨 핵심 — 학습이 normalization 된 데이터로 됐으니, 예측 입력도 똑같이 normalization

**모델은 학습할 때 사용된 데이터 분포를 기억**해. Logistic Regression은 평균 0, 표준편차 1인 값을 보고 가중치를 정했어. 그런데 새 분자의 원본 값(MolWt=200, logP=3.5 …)을 그대로 넣으면 모델 입장에서는 완전히 숫자라 예측 결과가 제대로 나오지 않음.

**예측 순서 (3단계, 순서 중요!)**
1. 학습 때 골랐던 **같은 feature**만 추출 (`features`)
2. 학습 때 fit한 **같은 scaler**로 `transform` ← ⚠️ Logistic Regression은 학습이 normalized로 됐으니 예측도 반드시 normalized
3. 모델로 `predict` / `predict_proba`

> ❌ **하지 말 것**: `StandardScaler().fit_transform(new_X)` — 새 평균/표준편차가 계산되어 학습 때와 다른 스케일이 되어 버림.
> ✅ **해야 할 것**: 저장된 `scaler.transform(new_X)` — 학습 때의 평균/표준편차를 그대로 적용.

In [6]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# 새로 예측할 분자 3개 (이름 + SMILES)
new_mols = {
    'Acetaminophen': 'CC(=O)Nc1ccc(O)cc1',
    'Aspirin':       'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine':      'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
}

# 분자별 descriptor 계산 → DataFrame으로 정리
rows = []
for name, smi in new_mols.items():
    mol = Chem.MolFromSmiles(smi)        # SMILES → Mol 객체
    desc = Descriptors.CalcMolDescriptors(mol)
    rows.append(desc)

# new_X = pd.DataFrame(rows, index=new_mols.keys(), columns=desc_names)
new_X = pd.DataFrame(rows)
                     
# 1) 학습 때 골랐던 같은 feature만 같은 순서로 추출
new_X_sel = new_X[features]

# 2) 학습 때 fit한 같은 scaler로 변환 (transform만! fit_transform 아님)
new_X_scaled = scaler.transform(new_X_sel)

# 3) 예측
new_pred = model.predict(new_X_scaled)
new_prob = model.predict_proba(new_X_scaled)

print('분자          :', list(new_mols.keys()))
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):')
print(new_prob.round(3))

분자          : ['Acetaminophen', 'Aspirin', 'Caffeine']
예측 label    : [np.int64(0), np.int64(0), np.int64(1)]
예측 확률(0,1):
[[0.954 0.046]
 [0.977 0.023]
 [0.301 0.699]]


---
# Part C. ✏️ 직접 해보기 — ANN, SVM, Tree, RF

Logistic Regression에서 했던 패턴 그대로 다른 모델도 저장해 봐.

**저장 패턴 요약**

```python
# normalization 필요한 모델 (LogReg, ANN, SVM)
saved = {'features': ..., 'scaler': ..., 'model': ...}

# normalization 불필요한 모델 (Tree, RF)
saved = {'features': ..., 'model': ...}    # scaler 없음
```

## ✏️ Quiz 1 — ANN (MLPClassifier) 저장

ANN은 normalization 필요 → **scaler도 같이 저장**.

**힌트**
- `from sklearn.neural_network import MLPClassifier`
- `MLPClassifier(hidden_layer_sizes=(??,), ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_ann.joblib`
- B-2 ~ B-4 패턴 그대로

In [10]:
import pandas as pd
import warnings
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

# 경고 무시
warnings.filterwarnings('ignore')

results = []

# 1. K값 후보 설정 (중복 제거 및 30, 40, 50 추가)
# 피처 총 개수가 50개보다 적을 경우를 대비해 유동적으로 설정하는 것이 좋습니다.
k_candidates = sorted(list(set(list(range(4, 25)) + [30, 40, 50])))

print("🚀 7개 모델 궁극의 Grid Search를 시작합니다. (random_state 해제 버전)")

for k in k_candidates:
    # 데이터셋의 피처 개수보다 k가 크면 에러가 나므로 예외 처리
    if k > X.shape[1]:
        continue
        
    print(f"▶ 현재 K = {k} 탐색 중...")
    
    # Feature Selection
    selector = SelectKBest(f_classif, k=k)
    selector.fit(X, y)
    cols = X.columns[selector.get_support()]
    
    # 데이터 준비
    X_sel_tree = X[cols] 
    X_sel_scaled = pd.DataFrame(StandardScaler().fit_transform(X_sel_tree), 
                                columns=cols, index=X_sel_tree.index)
    
    # ==========================================
    # 그룹 1: 스케일링 필수 모델 (선형/신경망)
    # ==========================================
    
    # (1) Logistic Regression
    for c in [0.1, 1, 10]:
        clf = LogisticRegression(C=c, max_iter=5000)
        train = clf.fit(X_sel_scaled, y).score(X_sel_scaled, y)
        cv = cross_val_score(clf, X_sel_scaled, y, cv=5).mean()
        results.append({'model': 'LogReg', 'k': k, 'param': f'C={c}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # (2) MLP
    hidden_layers = [(k,), (k, max(1, k // 2))] 
    for h in hidden_layers:
        for a in ['relu', 'tanh']:
            mlp = MLPClassifier(hidden_layer_sizes=h, activation=a, solver='lbfgs', max_iter=2000)
            train = mlp.fit(X_sel_scaled, y).score(X_sel_scaled, y)
            cv = cross_val_score(mlp, X_sel_scaled, y, cv=5).mean()
            results.append({'model': 'MLP', 'k': k, 'param': f'h={h},{a}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # (3) SVM
    for c in [1, 10]:
        for kf in ['linear', 'rbf']:
            svm = SVC(C=c, kernel=kf, gamma='scale')
            train = svm.fit(X_sel_scaled, y).score(X_sel_scaled, y)
            cv = cross_val_score(svm, X_sel_scaled, y, cv=5).mean()
            results.append({'model': 'SVM', 'k': k, 'param': f'C={c},{kf}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # ==========================================
    # 그룹 2: 트리 기반 모델 (random_state 제거)
    # ==========================================
    leaf_options = [1, 3, 9]
    n_options = [10, 50, 100] # n_estimators 후보

    # (4) Decision Tree
    for d in [3, 5, None]:
        for leaf in leaf_options:
            dt = DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf)
            train = dt.fit(X_sel_tree, y).score(X_sel_tree, y)
            cv = cross_val_score(dt, X_sel_tree, y, cv=5).mean()
            results.append({'model': 'DT', 'k': k, 'param': f'd={d},leaf={leaf}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # (5) Random Forest
    for n in n_options:
        for d in [3, 5, None]:
            for leaf in leaf_options:
                rf = RandomForestClassifier(n_estimators=n, max_depth=d, min_samples_leaf=leaf)
                train = rf.fit(X_sel_tree, y).score(X_sel_tree, y)
                cv = cross_val_score(rf, X_sel_tree, y, cv=5).mean()
                results.append({'model': 'RF', 'k': k, 'param': f'n={n},d={d},leaf={leaf}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # (6) XGBoost
    for n in n_options:
        for d in [3, 6]:
            for lr in [0.05, 0.1]:
                xgb = XGBClassifier(n_estimators=n, max_depth=d, learning_rate=lr, eval_metric='logloss')
                train = xgb.fit(X_sel_tree, y).score(X_sel_tree, y)
                cv = cross_val_score(xgb, X_sel_tree, y, cv=5).mean()
                results.append({'model': 'XGB', 'k': k, 'param': f'n={n},d={d},lr={lr}', 'train': round(train, 3), 'cv': round(cv, 3)})

    # (7) CatBoost
    for n in n_options:
        for d in [3, 6]:
            for lr in [0.05, 0.1]:
                cb = CatBoostClassifier(iterations=n, depth=d, learning_rate=lr, verbose=False)
                train = cb.fit(X_sel_tree, y).score(X_sel_tree, y)
                cv = cross_val_score(cb, X_sel_tree, y, cv=5).mean()
                results.append({'model': 'CatBoost', 'k': k, 'param': f'n={n},d={d},lr={lr}', 'train': round(train, 3), 'cv': round(cv, 3)})

print("\n🎉 모든 학습 완료!")

# 결과 저장 및 출력
df_ultimate = pd.DataFrame(results)
df_ultimate.to_csv('experiment_results_ultimate.csv', index=False)

# 각 모델별 CV 성능 기준 Best 1개씩 추출
best_of_best = df_ultimate.sort_values('cv', ascending=False).groupby('model').head(1).sort_values('cv', ascending=False)
print("\n[ 각 모델별 최적의 파라미터 및 성능 ]")
display(best_of_best)

🚀 7개 모델 궁극의 Grid Search를 시작합니다. (random_state 해제 버전)
▶ 현재 K = 4 탐색 중...
▶ 현재 K = 5 탐색 중...
▶ 현재 K = 6 탐색 중...
▶ 현재 K = 7 탐색 중...
▶ 현재 K = 8 탐색 중...
▶ 현재 K = 9 탐색 중...
▶ 현재 K = 10 탐색 중...
▶ 현재 K = 11 탐색 중...
▶ 현재 K = 12 탐색 중...
▶ 현재 K = 13 탐색 중...
▶ 현재 K = 14 탐색 중...
▶ 현재 K = 15 탐색 중...
▶ 현재 K = 16 탐색 중...
▶ 현재 K = 17 탐색 중...
▶ 현재 K = 18 탐색 중...
▶ 현재 K = 19 탐색 중...
▶ 현재 K = 20 탐색 중...
▶ 현재 K = 21 탐색 중...
▶ 현재 K = 22 탐색 중...
▶ 현재 K = 23 탐색 중...
▶ 현재 K = 24 탐색 중...
▶ 현재 K = 30 탐색 중...
▶ 현재 K = 40 탐색 중...
▶ 현재 K = 50 탐색 중...

🎉 모든 학습 완료!

[ 각 모델별 최적의 파라미터 및 성능 ]


,model,k,param,train,cv
665,RF,13,"n=10,d=None,leaf=1",0.974,0.875
1556,CatBoost,30,"n=50,d=6,lr=0.05",1.000,0.850
1047,XGB,18,"n=50,d=6,lr=0.05",0.923,0.850
1068,MLP,19,"h=(19,),relu",1.000,0.821
853,LogReg,16,C=1,0.897,0.821
512,DT,11,"d=5,leaf=3",0.897,0.800
791,SVM,15,"C=10,rbf",0.974,0.796


In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import joblib

# (1) 학습 — SelectKBest로 고른 cols, X_sel 그대로 사용
# scaler 새로 만들기 (학습 데이터에 fit)
scaler_ann = StandardScaler()
X_scaled_ann = scaler_ann.fit_transform(X_sel)

# MLPClassifier 학습
ann = MLPClassifier(hidden_layer_sizes=(3,), activation='logistic', max_iter=5000, random_state=0)
ann.fit(X_scaled_ann, y)
print('학습 정확도:', round(ann.score(X_scaled_ann, y), 3))


# (2) 저장 — features + scaler + model
# saved = {'features': ..., 'scaler': ..., 'model': ...}
# joblib.dump(saved, 'model_ann.joblib')
saved_ann = {
    'features': list(cols),  # 학습에 쓰인 10개의 변수 이름
    'scaler': scaler_ann,    # 학습 데이터의 기준(평균/표준편차)이 담긴 스케일러
    'model': ann             # 학습이 완료된 딥러닝 모델 본체
}
joblib.dump(saved_ann, 'model_ann.joblib')
print('\n✅ 저장 완료: model_ann.joblib')


# (3) 불러와서 정확도 확인
loaded_ann = joblib.load('model_ann.joblib')
features_ann = loaded_ann['features']
scaler_loaded = loaded_ann['scaler']
model_loaded = loaded_ann['model']

# 원래 데이터 X에서 저장해둔 10개 변수만 뽑고, 저장해둔 스케일러로 변환(transform만!)
X_check = X[features_ann]
X_check_scaled = scaler_loaded.transform(X_check)
y_pred_ann = model_loaded.predict(X_check_scaled)

print('불러온 모델 정확도:', round((y_pred_ann == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
new_X_sel = new_X[features_ann]
new_X_scaled = scaler_loaded.transform(new_X_sel) # 절대 fit_transform을 쓰면 안 됩니다!

new_pred_ann = model_loaded.predict(new_X_scaled)
new_prob_ann = model_loaded.predict_proba(new_X_scaled)

print('\n▶ 새로운 분자 예측 결과 (ANN)')
print('예측 label    :', list(new_pred_ann))
print('예측 확률(0,1):\n', new_prob_ann.round(3))

학습 정확도: 0.846

✅ 저장 완료: model_ann.joblib
불러온 모델 정확도: 0.846

▶ 새로운 분자 예측 결과 (ANN)
예측 label    : [np.int64(0), np.int64(0), np.int64(1)]
예측 확률(0,1):
 [[0.921 0.079]
 [0.931 0.069]
 [0.494 0.506]]


## ✏️ Quiz 2 — SVM 저장

SVM도 normalization 필요 → **scaler도 같이 저장**.

**힌트**
- `from sklearn.svm import SVC`
- `SVC(C=??, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_svm.joblib`

In [11]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import joblib

# (1) 학습  (K=15, C=3, kernel='rbf')
# 1-1. K=15로 최적의 특성 15개 선택
selector_svm = SelectKBest(f_classif, k=15)
selector_svm.fit(X, y)
cols_svm = X.columns[selector_svm.get_support()]
X_sel_svm = X[cols_svm]

# 1-2. scaler 새로 만들기 (학습 데이터에 fit)
scaler_svm = StandardScaler()
X_scaled_svm = scaler_svm.fit_transform(X_sel_svm)

# 1-3. SVM 학습 (새로운 분자 예측 확률을 보기 위해 probability=True 추가)
svm_best = SVC(C=3, kernel='rbf', probability=True, random_state=42)
svm_best.fit(X_scaled_svm, y)
print('학습 정확도:', round(svm_best.score(X_scaled_svm, y), 3))


# (2) 저장 — features + scaler + model → 'model_svm.joblib'
saved_svm = {
    'features': list(cols_svm), # 선택된 15개 특성 이름
    'scaler': scaler_svm,       # 15개 특성에 맞춰진 스케일러
    'model': svm_best           # 학습된 SVM 모델
}
joblib.dump(saved_svm, 'model_svm.joblib')
print('\n✅ 저장 완료: model_svm.joblib')


# (3) 불러와서 정확도 확인
loaded_svm = joblib.load('model_svm.joblib')
features_svm = loaded_svm['features']
scaler_loaded_svm = loaded_svm['scaler']
model_loaded_svm = loaded_svm['model']

# 원래 데이터 X에서 저장해둔 15개 변수만 뽑고, 스케일러로 변환(transform만!)
X_check_svm = X[features_svm]
X_check_scaled_svm = scaler_loaded_svm.transform(X_check_svm)
y_pred_svm = model_loaded_svm.predict(X_check_scaled_svm)

print('불러온 모델 정확도:', round((y_pred_svm == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
# 새로운 분자에서 모델이 필요로 하는 15개 변수만 추출하고 스케일링
new_X_sel_svm = new_X[features_svm]
new_X_scaled_svm = scaler_loaded_svm.transform(new_X_sel_svm) 

new_pred_svm = model_loaded_svm.predict(new_X_scaled_svm)
new_prob_svm = model_loaded_svm.predict_proba(new_X_scaled_svm)

print('\n▶ 새로운 분자 예측 결과 (최종 우승 모델: SVM)')
print('예측 label    :', list(new_pred_svm))
print('예측 확률(0,1):\n', new_prob_svm.round(3))

학습 정확도: 0.897

✅ 저장 완료: model_svm.joblib
불러온 모델 정확도: 0.897

▶ 새로운 분자 예측 결과 (최종 우승 모델: SVM)
예측 label    : [np.int64(0), np.int64(0), np.int64(0)]
예측 확률(0,1):
 [[0.907 0.093]
 [0.892 0.108]
 [0.748 0.252]]


## ✏️ Quiz 3 — Decision Tree 저장

트리는 normalization **불필요** → 저장할 때 **scaler 빼기**.
예측할 때도 `scaler.transform` 호출 ❌. `X_sel` 그대로 모델에 넣음.

**힌트**
- `from sklearn.tree import DecisionTreeClassifier`
- `DecisionTreeClassifier(max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_tree.joblib`
- 저장 딕셔너리: `{'features': ..., 'model': ...}` (scaler 없음)

In [12]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
# 최적의 특성 개수인 k=8로 특성을 선택합니다.
selector = SelectKBest(f_classif, k=8)
selector.fit(X, y)
cols_tree = X.columns[selector.get_support()]
X_sel = X[cols_tree]  # 스케일링을 전혀 하지 않은 원본 데이터 유지

# 이전 최적화에서 찾은 최적 파라미터 적용 (random_state는 제외)
tree_model = DecisionTreeClassifier(max_depth=None, min_samples_leaf=4)
tree_model.fit(X_sel, y)
print('학습 정확도:', round(tree_model.score(X_sel, y), 3))


# (2) 저장 — features + model만 → 'model_tree.joblib'
saved = {'features': list(cols_tree), 'model': tree_model}
joblib.dump(saved, 'model_tree.joblib')
print('\n✅ 저장 완료: model_tree.joblib')


# (3) 불러와서 정확도 확인
loaded_data = joblib.load('model_tree.joblib')
features = loaded_data['features']
model = loaded_data['model']

X_check = X[features]   # scaler.transform 없음! 원본 X에서 바로 추출
y_pred = model.predict(X_check)
print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
# new_X[features] 그대로 model.predict — scaler 없음
new_X_check = new_X[features] 
new_pred = model.predict(new_X_check)
new_prob = model.predict_proba(new_X_check)

print('\n▶ 새로운 분자 예측 결과 (Decision Tree)')
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):\n', new_prob.round(3))

학습 정확도: 0.872

✅ 저장 완료: model_tree.joblib
불러온 모델 정확도: 0.872

▶ 새로운 분자 예측 결과 (Decision Tree)
예측 label    : [np.int64(0), np.int64(0), np.int64(0)]
예측 확률(0,1):
 [[1.   0.  ]
 [0.75 0.25]
 [1.   0.  ]]


## ✏️ Quiz 4 — Random Forest 저장

Decision Tree와 동일하게 normalization **불필요**.

**힌트**
- `from sklearn.ensemble import RandomForestClassifier`
- `RandomForestClassifier(n_estimators=??, max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장 파일명: `model_rf.joblib`
- Tree와 같은 패턴 (scaler 없음)

In [13]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
# 최적의 특성 개수인 k=10으로 특성을 선택합니다.
selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
cols_rf = X.columns[selector.get_support()]
X_sel = X[cols_rf]  # 스케일링을 전혀 하지 않은 원본 데이터 유지

# 이전 최적화에서 찾은 최적 파라미터 적용 (n_estimators=100, max_depth=5)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5)
rf_model.fit(X_sel, y)
print('학습 정확도:', round(rf_model.score(X_sel, y), 3))


# (2) 저장 — features + model만 → 'model_rf.joblib'
saved = {'features': list(cols_rf), 'model': rf_model}
joblib.dump(saved, 'model_rf.joblib')
print('\n✅ 저장 완료: model_rf.joblib')


# (3) 불러와서 정확도 확인
loaded_data = joblib.load('model_rf.joblib')
features = loaded_data['features']
model = loaded_data['model']

X_check = X[features]   # scaler.transform 없음! 원본 X에서 바로 추출
y_pred = model.predict(X_check)
print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
# new_X[features] 그대로 model.predict — scaler 없음
new_X_check = new_X[features] 
new_pred = model.predict(new_X_check)
new_prob = model.predict_proba(new_X_check)

print('\n▶ 새로운 분자 예측 결과 (Random Forest)')
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):\n', new_prob.round(3))

학습 정확도: 1.0

✅ 저장 완료: model_rf.joblib
불러온 모델 정확도: 1.0

▶ 새로운 분자 예측 결과 (Random Forest)
예측 label    : [np.int64(0), np.int64(0), np.int64(0)]
예측 확률(0,1):
 [[0.94  0.06 ]
 [0.959 0.041]
 [0.813 0.187]]


## ✏️ Quiz 5 — XGBoost 저장

XGBoost도 tree 기반 모델 → normalization **불필요**.

⚠️ **XGBoost는 joblib 대신 자체 포맷(`.json`) 사용 권장**
- joblib(pickle)으로 저장하면 XGBoost 버전이 바뀔 때 호환성 문제가 생길 수 있음.
- XGBoost가 제공하는 `model.save_model('파일.json')` / `model.load_model('파일.json')`이 표준이고 버전 안전.
- 단, `save_model`은 **모델만** 저장 → `features`는 별도 파일로 따로 보관해야 함.

**힌트**
- `from xgboost import XGBClassifier`
- `XGBClassifier(n_estimators=??, max_depth=?, ...)` 지난 시간에 grid search 결과를 참고해서 가장 좋은 hyperparameter를 이용해서 훈련.
- 저장: `model.save_model('model_xgb.json')` + `joblib.dump(list(features), 'features_xgb.joblib')`
- 불러오기: `model = XGBClassifier(); model.load_model('model_xgb.json')`

In [14]:
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
# 최적의 특성 개수인 k=10으로 특성을 선택합니다.
selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
cols_xgb = X.columns[selector.get_support()]
X_sel = X[cols_xgb]  # 스케일링을 전혀 하지 않은 원본 데이터 유지

# 최적 파라미터 적용 (n_estimators=100, max_depth=3, learning_rate=0.1)
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, eval_metric='logloss')
xgb_model.fit(X_sel, y)
print('학습 정확도:', round(xgb_model.score(X_sel, y), 3))


# (2) 저장 — XGBoost 자체 포맷 (.json) 및 features 따로 저장
xgb_model.save_model('model_xgb.json')
joblib.dump(list(cols_xgb), 'features_xgb.joblib')
print('\n✅ 저장 완료: model_xgb.json (모델) & features_xgb.joblib (특성)')


# (3) 불러와서 정확도 확인
# ⚠️ 주의: 빈 모델을 먼저 선언한 뒤 load_model()을 사용해야 합니다.
loaded_xgb = XGBClassifier()
loaded_xgb.load_model('model_xgb.json')
features = joblib.load('features_xgb.joblib')

X_check = X[features]   # scaler.transform 없음! 원본 X에서 바로 추출
y_pred = loaded_xgb.predict(X_check)
print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
# new_X[features] 그대로 loaded_xgb.predict — scaler 없음
new_X_check = new_X[features] 
new_pred = loaded_xgb.predict(new_X_check)
new_prob = loaded_xgb.predict_proba(new_X_check)

print('\n▶ 새로운 분자 예측 결과 (XGBoost)')
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):\n', new_prob.round(3))

학습 정확도: 0.974

✅ 저장 완료: model_xgb.json (모델) & features_xgb.joblib (특성)
불러온 모델 정확도: 0.974

▶ 새로운 분자 예측 결과 (XGBoost)
예측 label    : [np.int64(0), np.int64(0), np.int64(0)]
예측 확률(0,1):
 [[0.97  0.03 ]
 [0.926 0.074]
 [0.76  0.24 ]]


In [15]:
from sklearn.feature_selection import SelectKBest, f_classif
from catboost import CatBoostClassifier
import joblib

# (1) 학습 — X_sel 그대로 (scaling 없이)
# K=4로 특성을 선택합니다.
selector = SelectKBest(f_classif, k=4)
selector.fit(X, y)
cols_cb = X.columns[selector.get_support()]
X_sel = X[cols_cb]  # 스케일링을 전혀 하지 않은 원본 데이터 유지

# 최적 파라미터 적용 (이전 결과 참고: 깊이, 학습률 등 세팅)
cb_model = CatBoostClassifier(iterations=100, depth=5, learning_rate=0.05, verbose=False)
cb_model.fit(X_sel, y)
print('학습 정확도:', round(cb_model.score(X_sel, y), 3))


# (2) 저장 — CatBoost 자체 포맷 (.cbm) 및 features 따로 저장
cb_model.save_model('model_catboost.cbm')
joblib.dump(list(cols_cb), 'features_catboost.joblib')
print('\n✅ 저장 완료: model_catboost.cbm (모델) & features_catboost.joblib (특성)')


# (3) 불러와서 정확도 확인
# ⚠️ 주의: 빈 모델을 먼저 선언한 뒤 load_model()을 사용해야 합니다.
loaded_cb = CatBoostClassifier()
loaded_cb.load_model('model_catboost.cbm')
features_cb = joblib.load('features_catboost.joblib')

X_check = X[features_cb]   # scaler.transform 없음! 원본 X에서 바로 추출
y_pred = loaded_cb.predict(X_check)
print('불러온 모델 정확도:', round((y_pred == y).mean(), 3))


# (4) 마지막 3개 분자에 예측
# new_X[features] 그대로 loaded_cb.predict — scaler 없음
new_X_check = new_X[features_cb] 
new_pred = loaded_cb.predict(new_X_check)
new_prob = loaded_cb.predict_proba(new_X_check)

print('\n▶ 새로운 분자 예측 결과 (CatBoost)')
print('예측 label    :', list(new_pred))
print('예측 확률(0,1):\n', new_prob.round(3))

학습 정확도: 0.897

✅ 저장 완료: model_catboost.cbm (모델) & features_catboost.joblib (특성)
불러온 모델 정확도: 0.897

▶ 새로운 분자 예측 결과 (CatBoost)
예측 label    : [np.int64(0), np.int64(0), np.int64(0)]
예측 확률(0,1):
 [[0.677 0.323]
 [0.824 0.176]
 [0.607 0.393]]


---
## 정리 — 기계학습 모델 개발의 최종 결과물

**저장 / 불러오기 핵심 명령**
```python
joblib.dump(객체, '파일.joblib')   # 저장 (대부분 모델)
객체 = joblib.load('파일.joblib')   # 불러오기

# XGBoost는 자체 포맷 (.json) 사용
model.save_model('파일.json')
model.load_model('파일.json')
```

**같이 저장해야 할 것**
1. `features` — SelectKBest로 고른 컬럼 이름 (새 분자에서 같은 순서로 추출)
2. `scaler` — normalization 쓰는 모델만 (LogReg / ANN / SVM)
3. `model` — 학습된 모델 본체

**새 분자 예측 3단계**
1. `new_X[features]` — 같은 feature만 같은 순서로
2. `scaler.transform(...)` — ⚠️ `fit_transform` 아니고 **`transform`**
3. `model.predict(...)`

**모델별 정리**

| 모델 | 저장 포맷 | scaler 저장? | 예측 시 transform 호출? |
|---|---|---|---|
| Logistic Regression | `.joblib` | ✅ | ✅ |
| ANN (MLP) | `.joblib` | ✅ | ✅ |
| SVM | `.joblib` | ✅ | ✅ |
| Decision Tree | `.joblib` | ❌ | ❌ |
| Random Forest | `.joblib` | ❌ | ❌ |
| XGBoost | `.json` (+ features는 별도 `.joblib`) | ❌ | ❌ |

> ✅ 5개 quiz를 다 끝냈다면, 디렉터리에 `model_logreg.joblib`, `model_ann.joblib`, `model_svm.joblib`, `model_tree.joblib`, `model_rf.joblib`, `model_xgb.json`, `features_xgb.joblib` 일곱 개 파일이 생겨 있어야 함.

기말고사 할 때 random state 있으면 감점.